In [2]:
import pandas as pd

df=pd.read_excel("/home/rishi/ML Projects/Air Pollution/站点列表-2022.02.13起.xlsx")
df

,监测点编码,监测点名称,城市,经度,纬度,对照点
0,1001A,万寿西宫,北京,116.3621,39.8784,N
1,1002A,定陵(对照点),北京,116.2202,40.2915,Y
2,1003A,东四,北京,116.4174,39.9289,N
3,1004A,天坛,北京,116.4072,39.8863,N
4,1005A,农展馆,北京,116.462,39.9365,N
...,...,...,...,...,...,...
2021,3734A,理想之城,东营,118.35,37.26,N
2022,3860A,皇山花园小区,临沂,118.392,35.03,N
2023,3861A,寒亭区实验一小,潍坊,119.2041,36.7699,N
2024,3866A,三高中,营口,122.2469,40.66302,N


In [3]:
import os, re

imputed_dir = "/home/rishi/ML Projects/Air Pollution/CNEMC/imputed"
all_pollutants = {"CO", "NO2", "Ozone", "PM10", "PM2.5", "SO2"}

# Parse site IDs and pollutants from filenames
site_pollutants = {}
for f in os.listdir(imputed_dir):
    m = re.match(r"site_(.+?)_(.+)\.csv", f)
    if m:
        site_id, pollutant = m.group(1), m.group(2)
        site_pollutants.setdefault(site_id, set()).add(pollutant)

# Build summary dataframe
rows = [{"site_id": sid, "has_all_pollutants": pols == all_pollutants, "pollutants_present": sorted(pols), "n_pollutants": len(pols)} for sid, pols in site_pollutants.items()]
sites_df = pd.DataFrame(rows)

# Left join with lat/long from the station list
sites_df = sites_df.merge(
    df.rename(columns={"监测点编码": "site_id", "经度": "longitude", "纬度": "latitude", "监测点名称": "site_name", "城市": "city"})[["site_id", "site_name", "city", "longitude", "latitude"]],
    on="site_id", how="left"
)

print(f"Total sites in imputed folder: {len(sites_df)}")
print(f"Sites with all 6 pollutants: {sites_df['has_all_pollutants'].sum()}")
print(f"Sites with lat/long match: {sites_df['latitude'].notna().sum()}")
sites_df

Total sites in imputed folder: 1491
Sites with all 6 pollutants: 1451
Sites with lat/long match: 1491


,site_id,has_all_pollutants,pollutants_present,n_pollutants,site_name,city,longitude,latitude
0,3664A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,连云街道,连云港,119.4172,34.7341
1,3376A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,南苑二区,曲靖,103.7947,25.462
2,3473A,False,"[CO, NO2, Ozone, PM10, SO2]",5,技师学院,开封,114.2485,34.8306
3,1876A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,河东子站,三亚,109.5078,18.2489
4,2669A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,州政府,甘南州,102.9099,34.9841
...,...,...,...,...,...,...,...,...
1486,2642A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,香溪洞(对照点),安康,109.0318,32.6537
1487,1277A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,包河区,合肥,117.3027,31.7964
1488,3003A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,经开区,常州,120.0522,31.7748
1489,2907A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,吴兴区站,湖州,120.1844,30.8596


In [6]:
import numpy as np

# Filter to sites with all pollutants and valid lat/long
complete = sites_df[sites_df["has_all_pollutants"] & sites_df["latitude"].notna()].copy()
print(f"Sites with all pollutants + lat/long: {len(complete)}")

# Spatially stratified random sampling: grid the bounding box into cells, sample proportionally
n_target = 200
n_grid = 15  # 15x15 grid


lon_bins = np.linspace(complete["longitude"].min(), complete["longitude"].max(), n_grid + 1)
lat_bins = np.linspace(complete["latitude"].min(), complete["latitude"].max(), n_grid + 1)

complete["lon_bin"] = pd.cut(complete["longitude"], bins=lon_bins, labels=False, include_lowest=True)
complete["lat_bin"] = pd.cut(complete["latitude"], bins=lat_bins, labels=False, include_lowest=True)
complete["grid_cell"] = complete["lon_bin"].astype(str) + "_" + complete["lat_bin"].astype(str)

# Proportional allocation: each cell gets samples proportional to its share of sites (min 1 if occupied)
cell_counts = complete["grid_cell"].value_counts()
n_cells = len(cell_counts)
alloc = (cell_counts / cell_counts.sum() * n_target).apply(np.floor).astype(int).clip(lower=1)

# Adjust to hit exactly n_target
while alloc.sum() > n_target:
    idx = alloc[alloc > 1].idxmax()  # only decrement cells above 1
    alloc[idx] -= 1

while alloc.sum() < n_target:
    # add to largest cells first (most underrepresented)
    remaining = cell_counts[cell_counts > alloc]
    if remaining.empty:
        break
    idx = remaining.idxmax()
    alloc[idx] += 1

np.random.seed(42)
sampled = complete.groupby("grid_cell", group_keys=False).apply(
    lambda g: g.sample(n=min(alloc.get(g.name, 0), len(g)), random_state=42)
)

sampled = sampled.drop(columns=["lon_bin", "lat_bin"])
print(f"Sampled sites: {len(sampled)}")
sampled

Sites with all pollutants + lat/long: 1451
Sampled sites: 200


,site_id,has_all_pollutants,pollutants_present,n_pollutants,site_name,city,longitude,latitude
971,3665A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,克州文化中心,克孜勒苏柯尔克孜自治州,76.1658,39.7157
1370,2698A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,巡警大队,喀什地区,76.0072,39.4467
1299,1379A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,华柏园,中山,113.3769,22.5211
561,3566A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,图书馆,大同,113.1186,40.2556
999,1007A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,海淀万柳,北京,116.2878,39.9611
...,...,...,...,...,...,...,...,...
1238,3636A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,英才小学,洛阳,112.4388,34.6781
688,1733A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,临钢医院,临汾,111.5531,36.0783
1053,2174A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,榆次区政府,晋中,112.6991,37.6982
616,2167A,True,"[CO, NO2, Ozone, PM10, PM2.5, SO2]",6,朔唯,朔州,112.4575,39.3617


In [7]:
import shutil

out_dir = "/home/rishi/ML Projects/Air Pollution/CNEMC/cnemc_sampled_200"
os.makedirs(out_dir, exist_ok=True)

copied = 0
for site_id in sampled["site_id"]:
    for f in os.listdir(imputed_dir):
        if f.startswith(f"site_{site_id}_"):
            shutil.copy2(os.path.join(imputed_dir, f), os.path.join(out_dir, f))
            copied += 1

print(f"Copied {copied} files for {len(sampled)} sites to {out_dir}")

Copied 1200 files for 200 sites to /home/rishi/ML Projects/Air Pollution/CNEMC/cnemc_sampled_200
